# Forecast Error Analysis

This notebook analyzes January 2024 UK wind forecasts from `WINDFOR` against `FUELHH` actuals.

Method:
- pull January 2024 actual wind generation from `FUELHH`
- pull `WINDFOR` publishes from 48 hours before January through the end of January
- for each target time and chosen horizon, pick the latest forecast with `publishTime <= targetTime - horizon`
- compute absolute error and summarize it by horizon and time of day

The code is written explicitly so the assumptions and selection logic are inspectable.

In [ ]:
from datetime import datetime, timedelta, timezone\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport requests\nimport seaborn as sns\n\nsns.set_theme(style='whitegrid')\n\nUTC = timezone.utc\nBASE_URL = 'https://data.elexon.co.uk/bmrs/api/v1'\nJAN_START = datetime(2024, 1, 1, tzinfo=UTC)\nJAN_END = datetime(2024, 2, 1, tzinfo=UTC)\nFORECAST_FETCH_START = JAN_START - timedelta(hours=48)\nHORIZONS = [0, 4, 8, 12, 24, 36, 48]\n

In [ ]:
def fetch_dataset(path: str, params: dict) -> pd.DataFrame:\n    response = requests.get(f'{BASE_URL}{path}', params={**params, 'format': 'json'}, timeout=120)\n    response.raise_for_status()\n    payload = response.json()\n    if isinstance(payload, dict) and 'data' in payload:\n        payload = payload['data']\n    return pd.DataFrame(payload)\n\n\nactuals = fetch_dataset(\n    '/datasets/FUELHH/stream',\n    {\n        'settlementDateFrom': '2024-01-01',\n        'settlementDateTo': '2024-01-31',\n        'fuelType': 'WIND',\n    },\n)\nactuals['startTime'] = pd.to_datetime(actuals['startTime'], utc=True)\nactuals = actuals[['startTime', 'generation']].rename(columns={'generation': 'actualGeneration'})\n\nforecasts = fetch_dataset(\n    '/datasets/WINDFOR/stream',\n    {\n        'publishDateTimeFrom': FORECAST_FETCH_START.isoformat().replace('+00:00', 'Z'),\n        'publishDateTimeTo': (JAN_END - timedelta(minutes=1)).isoformat().replace('+00:00', 'Z'),\n    },\n)\nforecasts['startTime'] = pd.to_datetime(forecasts['startTime'], utc=True)\nforecasts['publishTime'] = pd.to_datetime(forecasts['publishTime'], utc=True)\nforecasts['horizonHours'] = (forecasts['startTime'] - forecasts['publishTime']).dt.total_seconds() / 3600\nforecasts = forecasts[(forecasts['horizonHours'] >= 0) & (forecasts['horizonHours'] <= 48)].copy()\nactuals.head(), forecasts.head()\n

In [ ]:
def merge_for_horizon(horizon_hours: int) -> pd.DataFrame:\n    merged_rows = []\n    forecasts_by_target = {k: v.sort_values('publishTime') for k, v in forecasts.groupby('startTime')}\n\n    for row in actuals.itertuples(index=False):\n        cutoff = row.startTime - timedelta(hours=horizon_hours)\n        options = forecasts_by_target.get(row.startTime)\n        selected = None\n        if options is not None:\n            eligible = options[options['publishTime'] <= cutoff]\n            if not eligible.empty:\n                selected = eligible.iloc[-1]\n\n        forecast_generation = None if selected is None else float(selected['generation'])\n        publish_time = None if selected is None else selected['publishTime']\n        absolute_error = None if selected is None else abs(float(row.actualGeneration) - forecast_generation)\n\n        merged_rows.append({\n            'startTime': row.startTime,\n            'actualGeneration': float(row.actualGeneration),\n            'forecastGeneration': forecast_generation,\n            'forecastPublishTime': publish_time,\n            'requestedHorizonHours': horizon_hours,\n            'effectiveHorizonHours': None if selected is None else (row.startTime - publish_time).total_seconds() / 3600,\n            'absoluteError': absolute_error,\n        })\n\n    merged = pd.DataFrame(merged_rows)\n    merged['hourOfDay'] = merged['startTime'].dt.hour\n    return merged\n\n\nmerged_by_horizon = pd.concat([merge_for_horizon(h) for h in HORIZONS], ignore_index=True)\nerror_rows = merged_by_horizon.dropna(subset=['absoluteError']).copy()\n\nsummary = error_rows.groupby('requestedHorizonHours')['absoluteError'].agg(\n    mean='mean',\n    median='median',\n    p99=lambda s: s.quantile(0.99),\n    count='count',\n).reset_index()\nsummary\n

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))\n\nsns.lineplot(data=summary, x='requestedHorizonHours', y='mean', marker='o', ax=axes[0], label='Mean absolute error')\nsns.lineplot(data=summary, x='requestedHorizonHours', y='median', marker='o', ax=axes[0], label='Median absolute error')\nsns.lineplot(data=summary, x='requestedHorizonHours', y='p99', marker='o', ax=axes[0], label='P99 absolute error')\naxes[0].set_title('Error vs horizon')\naxes[0].set_xlabel('Requested horizon (hours)')\naxes[0].set_ylabel('Absolute error (MW)')\n\ntod = error_rows.groupby(['requestedHorizonHours', 'hourOfDay'])['absoluteError'].mean().reset_index()\nsns.lineplot(data=tod, x='hourOfDay', y='absoluteError', hue='requestedHorizonHours', palette='viridis', ax=axes[1])\naxes[1].set_title('Mean absolute error by hour of day')\naxes[1].set_xlabel('Hour of day (UTC)')\naxes[1].set_ylabel('Absolute error (MW)')\n\nplt.tight_layout()\nplt.show()\n\nsummary.style.format({'mean': '{:,.0f}', 'median': '{:,.0f}', 'p99': '{:,.0f}'})\n